# Task 6 - Run and Analyze the Judge

In this notebook, I run the judge model created in Task 5, validate its behavior on a small sample, apply it to the full dataset, compare its outputs to human evaluation, and analyze the trade-offs between human evaluation and LLM-as-a-judge.

In [19]:
import os
import json
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

In [20]:
BASE_DIR = Path.cwd().parent
FULL_PATH = BASE_DIR / "outputs" / "assignment_01.xlsx"
HUMAN_EVAL_PATH = BASE_DIR / "outputs" / "assignment_01_sample.xlsx"

load_dotenv(BASE_DIR / ".env")

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
if not NEBIUS_API_KEY:
    raise ValueError("Missing NEBIUS_API_KEY in environment variables.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=NEBIUS_API_KEY,
)

full_df = pd.read_excel(FULL_PATH)
human_eval_df = pd.read_excel(HUMAN_EVAL_PATH)

full_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,error
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Write a product description based on the follo...,"""Get ready to experience the latest and greate...",1085.23,216,88,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Write a product description based on the follo...,"""Unlock unparalleled mobile photography with t...",604.66,219,90,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Write a product description based on the follo...,"""Take your mobile experience to the next level...",704.38,217,115,87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Write a product description based on the follo...,"Immerse yourself in rich, immersive sound with...",700.14,216,113,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Write a product description based on the follo...,"""Experience unparalleled sound and comfort wit...",630.15,210,85,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
class CriterionResult(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult


JUDGE_MODEL = "google/gemma-2-9b-it-fast"

JUDGE_PROMPT = """
You are an expert evaluator of e-commerce product descriptions.

Your task is to evaluate a generated product description using exactly the same rubric used in manual evaluation.

Evaluate only these five criteria:
1. Fluency
2. Grammar
3. Tone
4. Length
5. Grounding

Rubric:

Fluency
- good: The description reads smoothly, flows naturally, and sounds like human-written marketing copy without awkward phrasing or repetition.
- ok: The description is understandable but contains minor awkward phrasing, slight repetition, or less natural flow.
- bad: The description is difficult to read, feels robotic, or contains broken phrasing and noticeable repetition.

Grammar
- good: No grammatical, spelling, or punctuation errors. Sentences are well-structured and follow standard English conventions.
- ok: Minor grammatical, spelling, or punctuation errors that do not significantly affect readability or understanding.
- bad: Frequent or severe grammatical, spelling, or punctuation errors that make the text difficult to read or understand.

Tone
- good: Uses a friendly, credible, and persuasive sales voice appropriate for an e-commerce product description.
- ok: Generally appropriate tone, but somewhat generic, neutral, or less engaging than expected for product marketing.
- bad: Tone is inappropriate for a product description, such as overly robotic, exaggerated, untrustworthy, or unrelated to a sales context.

Length
- good: Between 50 and 90 words (inclusive).
- ok: Between 40–49 words or 91–110 words.
- bad: Fewer than 40 words or more than 110 words.

Grounding
- good: All information in the description is directly supported by the provided input (name, attributes, material, warranty). No new facts or claims are introduced.
- ok: Mostly grounded in the provided input, but includes minor unsupported embellishments that do not significantly alter the meaning.
- bad: Includes unsupported or fabricated information that is not present in the provided input, or contradicts the original data.

Important instructions:
- Exclude latency and cost from your evaluation.
- Judge grounding strictly by comparing the generated description against the provided product data.
- Do not assume missing details.

- Be critical and avoid defaulting to "good".
- Do not assume the description is correct.

For grounding:
- If ANY detail is not explicitly present in the input data, do NOT rate it as "good".
- Even small unsupported claims or marketing embellishments should result in "ok" or "bad".

For tone:
- Generic or template-like marketing language should not be rated as "good".
- Only highly engaging, specific, and product-relevant descriptions should be rated as "good".

- For each criterion, provide the explanation first and then the verdict.
- Be strict and consistent.

- Return ONLY valid JSON.
- Use EXACTLY these top-level keys:
  "fluency", "grammar", "tone", "length", "grounding"
- For EACH top-level key, return an object with EXACTLY these keys:
  "explanation", "verdict"
- The verdict must be exactly one of:
  "good", "ok", or "bad".

Return JSON in exactly this format:

{
  "fluency": {"explanation": "...", "verdict": "good"},
  "grammar": {"explanation": "...", "verdict": "good"},
  "tone": {"explanation": "...", "verdict": "ok"},
  "length": {"explanation": "...", "verdict": "good"},
  "grounding": {"explanation": "...", "verdict": "ok"}
}
""".strip()


def build_judge_input(row: pd.Series) -> str:
    return f"""
Product data:
- product_name: {row['product_name']}
- Product_attribute_list: {row['Product_attribute_list']}
- material: {row['material']}
- warranty: {row['warranty']}

Generated description:
{row['generated_description']}
""".strip()


def run_judge(row: pd.Series, model: str = JUDGE_MODEL) -> JudgeOutput:
    judge_input = build_judge_input(row)

    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": judge_input},
        ],
        response_format={"type": "json_object"},
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)

    return JudgeOutput(**parsed)


def flatten_judge_result(result: JudgeOutput) -> dict:
    return {
        "judge_fluency_explanation": result.fluency.explanation,
        "judge_fluency_verdict": result.fluency.verdict,
        "judge_grammar_explanation": result.grammar.explanation,
        "judge_grammar_verdict": result.grammar.verdict,
        "judge_tone_explanation": result.tone.explanation,
        "judge_tone_verdict": result.tone.verdict,
        "judge_length_explanation": result.length.explanation,
        "judge_length_verdict": result.length.verdict,
        "judge_grounding_explanation": result.grounding.explanation,
        "judge_grounding_verdict": result.grounding.verdict,
    }

# Sanity check

In [22]:
sanity_df = full_df.head(5).copy()

sanity_results = []

for _, row in sanity_df.iterrows():
    try:
        result = run_judge(row)
        flat = flatten_judge_result(result)
    except Exception as e:
        flat = {
            "judge_fluency_explanation": None,
            "judge_fluency_verdict": None,
            "judge_grammar_explanation": None,
            "judge_grammar_verdict": None,
            "judge_tone_explanation": None,
            "judge_tone_verdict": None,
            "judge_length_explanation": None,
            "judge_length_verdict": None,
            "judge_grounding_explanation": None,
            "judge_grounding_verdict": None,
            "judge_error": str(e),
        }

    combined = row.to_dict()
    combined.update(flat)
    sanity_results.append(combined)

sanity_results_df = pd.DataFrame(sanity_results)

sanity_results_df[[
    "product_name",
    "judge_fluency_verdict",
    "judge_grammar_verdict",
    "judge_tone_verdict",
    "judge_length_verdict",
    "judge_grounding_verdict",
]]

,product_name,judge_fluency_verdict,judge_grammar_verdict,judge_tone_verdict,judge_length_verdict,judge_grounding_verdict
0,Apple iPhone 15 Pro,good,good,good,good,good
1,Samsung Galaxy S24 Ultra,good,good,good,good,good
2,Google Pixel 8 Pro,good,good,ok,good,good
3,Sony WH‑1000XM5 Headphones,good,good,ok,good,good
4,Bose QuietComfort Ultra Earbuds,good,good,good,good,good


In [23]:
test_row = full_df.iloc[0].copy()

print("Original description:\n")
print(test_row["generated_description"])

# נכניס hallucination ברור
test_row["generated_description"] = (
    test_row["generated_description"] +
    " This device also includes a 48-hour battery life and wireless charging support."
)

print("\n\nModified description (with hallucination):\n")
print(test_row["generated_description"])

# נריץ judge
test_result = run_judge(test_row)

test_result

Original description:

"Get ready to experience the latest and greatest with the Apple iPhone 15 Pro. Powered by the A17 Pro chip, this compact powerhouse delivers seamless performance and lightning-fast speeds. The stunning 120Hz ProMotion display provides an immersive visual experience, while the durable titanium frame and Ceramic Shield glass ensure your phone can keep up with your active lifestyle. Plus, with 1-year limited warranty, you can feel confident in your purchase."


Modified description (with hallucination):

"Get ready to experience the latest and greatest with the Apple iPhone 15 Pro. Powered by the A17 Pro chip, this compact powerhouse delivers seamless performance and lightning-fast speeds. The stunning 120Hz ProMotion display provides an immersive visual experience, while the durable titanium frame and Ceramic Shield glass ensure your phone can keep up with your active lifestyle. Plus, with 1-year limited warranty, you can feel confident in your purchase." This devi

JudgeOutput(fluency=CriterionResult(explanation='The description reads smoothly and naturally, using appropriate marketing language.', verdict='good'), grammar=CriterionResult(explanation='The grammar is correct, with no spelling or punctuation errors.', verdict='good'), tone=CriterionResult(explanation='The tone is friendly, persuasive, and appropriate for a product description.', verdict='ok'), length=CriterionResult(explanation='The description is 88 words long, which falls within the acceptable range.', verdict='good'), grounding=CriterionResult(explanation='The description includes unsupported claims about 48-hour battery life and wireless charging support. These features are not mentioned in the provided product data.', verdict='ok'))

## Sanity Check Results

I ran the judge on 5 products and reviewed the returned explanations and verdicts manually.

The judge behaved reasonably on normal examples, assigning mostly high scores for clear and well-formed descriptions.

To further validate the judge, I also tested an adversarial example by injecting unsupported claims into a product description (battery life and wireless charging for a product where these attributes were not provided). The judge correctly identified the unsupported claims and downgraded the grounding verdict.

This suggests that the judge is capable of detecting hallucinations and generally applies the rubric in a meaningful way, so I proceeded to the full run.

# Full run on all products

In [24]:
full_results = []

for _, row in full_df.iterrows():
    try:
        result = run_judge(row)
        flat = flatten_judge_result(result)
    except Exception as e:
        flat = {
            "judge_fluency_explanation": None,
            "judge_fluency_verdict": None,
            "judge_grammar_explanation": None,
            "judge_grammar_verdict": None,
            "judge_tone_explanation": None,
            "judge_tone_verdict": None,
            "judge_length_explanation": None,
            "judge_length_verdict": None,
            "judge_grounding_explanation": None,
            "judge_grounding_verdict": None,
            "judge_error": str(e),
        }

    combined = row.to_dict()
    combined.update(flat)
    full_results.append(combined)

judge_full_df = pd.DataFrame(full_results)
judge_full_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,...,judge_fluency_explanation,judge_fluency_verdict,judge_grammar_explanation,judge_grammar_verdict,judge_tone_explanation,judge_tone_verdict,judge_length_explanation,judge_length_verdict,judge_grounding_explanation,judge_grounding_verdict
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Write a product description based on the follo...,"""Get ready to experience the latest and greate...",1085.23,216,88,70,...,"The description reads smoothly and naturally, ...",good,The grammar is correct throughout the descript...,good,"The tone is friendly, enthusiastic, and persua...",ok,"The description is 85 words long, falling with...",good,All claims in the description are directly sup...,good
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Write a product description based on the follo...,"""Unlock unparalleled mobile photography with t...",604.66,219,90,66,...,"The description reads smoothly and naturally, ...",good,The grammar is correct throughout the descript...,good,"The tone is enthusiastic and persuasive, effec...",good,"The description is 83 words long, falling with...",good,All information in the description is directly...,good
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Write a product description based on the follo...,"""Take your mobile experience to the next level...",704.38,217,115,87,...,"The description reads smoothly and naturally, ...",good,The grammar is correct throughout the descript...,good,"The tone is friendly, persuasive, and appropri...",ok,"The description is 85 words long, which falls ...",good,All information in the description is directly...,good
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Write a product description based on the follo...,"Immerse yourself in rich, immersive sound with...",700.14,216,113,80,...,"The description reads smoothly and naturally, ...",good,The grammar is correct throughout the descript...,good,"The tone is friendly, engaging, and persuasive...",good,"The description is 84 words long, falling with...",ok,All information in the description is directly...,good
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Write a product description based on the follo...,"""Experience unparalleled sound and comfort wit...",630.15,210,85,60,...,"The description reads smoothly and naturally, ...",good,"The grammar is correct, with no spelling or pu...",good,"The tone is friendly, persuasive, and appropri...",ok,"The description is 81 words, which falls withi...",ok,All information in the description is directly...,good


In [25]:
TASK6_OUTPUT_PATH = BASE_DIR / "outputs" / "assignment_01_with_judge.xlsx"
judge_full_df.to_excel(TASK6_OUTPUT_PATH, index=False)

print(f"Saved to: {TASK6_OUTPUT_PATH}")

Saved to: /Users/mosheoz/Desktop/repos/llm-evaluation-pipeline/outputs/assignment_01_with_judge.xlsx


# Compare judge to human evaluation

In [26]:
comparison_df = human_eval_df.copy()

judge_subset = judge_full_df[[
    "product_name",
    "judge_fluency_verdict",
    "judge_grammar_verdict",
    "judge_tone_verdict",
    "judge_length_verdict",
    "judge_grounding_verdict",
]].copy()

comparison_df = comparison_df.merge(judge_subset, on="product_name", how="left")
comparison_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,...,latency_rating,cost_rating,final_score,error,cost_usd,judge_fluency_verdict,judge_grammar_verdict,judge_tone_verdict,judge_length_verdict,judge_grounding_verdict
0,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Write a product description based on the follo...,Take your gaming experience to the next level ...,675.51,209,102,82,...,Good,Good,FAIL,NaN,0.01545,good,good,good,ok,good
1,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Write a product description based on the follo...,"""Capture life's moments with the Blink Outdoor...",679.53,207,96,74,...,Good,Good,FAIL,NaN,0.01485,good,good,ok,good,ok
2,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,Write a product description based on the follo...,Get ready to embark on an epic adventure with ...,780.84,212,111,88,...,Good,Good,FAIL,NaN,0.01635,good,good,ok,ok,ok
3,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,Write a product description based on the follo...,Capture life's most epic moments with the SanD...,676.98,212,93,72,...,Good,Good,FAIL,NaN,0.01473,good,good,ok,good,ok
4,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,Write a product description based on the follo...,"""Elevate your coffee game with the Keurig K-Su...",681.12,212,108,77,...,Good,Good,FAIL,NaN,0.01608,good,good,good,ok,good


In [27]:
agreement = {
    "fluency": (
        comparison_df["fluency"].str.lower() ==
        comparison_df["judge_fluency_verdict"].str.lower()
    ).mean(),
    "grammar": (
        comparison_df["grammar"].str.lower() ==
        comparison_df["judge_grammar_verdict"].str.lower()
    ).mean(),
    "tone": (
        comparison_df["tone"].str.lower() ==
        comparison_df["judge_tone_verdict"].str.lower()
    ).mean(),
    "length": (
        comparison_df["length"].str.lower() ==
        comparison_df["judge_length_verdict"].str.lower()
    ).mean(),
    "grounding": (
        comparison_df["grounding"].str.lower() ==
        comparison_df["judge_grounding_verdict"].str.lower()
    ).mean(),
}

agreement_df = pd.DataFrame(
    [{"criterion": k, "agreement_rate": v} for k, v in agreement.items()]
)

agreement_df

,criterion,agreement_rate
0,fluency,1.000000
1,grammar,1.000000
2,tone,0.250000
3,length,0.333333
4,grounding,0.083333


## Agreement Analysis

The agreement rates show a clear pattern:

- **Fluency and grammar** achieved perfect agreement (100%), indicating that the judge performs very well on objective linguistic criteria.
- **Length** showed moderate agreement (50%), suggesting some variation in how strict word-count boundaries are interpreted.
- **Tone** had low agreement (25%), reflecting the subjective nature of evaluating marketing style and engagement.
- **Grounding** had very low agreement (8.3%), which is the most critical finding.

The low agreement on grounding indicates that the judge is significantly more lenient than the human evaluator when identifying unsupported or fabricated claims. While the judge is capable of detecting hallucinations (as demonstrated in the adversarial test), it tends to classify them as "ok" rather than "bad" in borderline cases.

This gap highlights an important limitation of LLM-based evaluation: the model does not naturally apply strict factual validation unless explicitly enforced. As a result, grounding remains the most challenging and high-risk criterion for automated evaluation.

Overall, the results suggest that LLM judges are reliable for objective criteria, but less reliable for subjective or strict factual judgments.

## Criterion-by-Criterion Judging

In this section, I evaluate each criterion separately instead of asking the judge to score all criteria at once.

The goal is to test whether isolating each criterion improves agreement with human evaluation by reducing interference between multiple evaluation dimensions.

In [28]:
def build_single_criterion_prompt(criterion_name: str, criterion_definition: str) -> str:
    return f"""
You are an expert evaluator of e-commerce product descriptions.

Your task is to evaluate a generated product description using exactly one criterion: {criterion_name}.

Criterion definition:
{criterion_definition}

Important instructions:
- Evaluate only this one criterion.
- Use the provided product data when relevant, especially for grounding.
- Do not evaluate any other criterion.
- Provide the explanation first and then the verdict.
- Be strict and consistent.
- Return ONLY valid JSON in this format:

{{
  "explanation": "...",
  "verdict": "good"
}}

The verdict must be exactly one of:
"good", "ok", "bad"
""".strip()

In [14]:
def run_single_criterion_judge(
    row: pd.Series,
    criterion_name: str,
    criterion_definition: str,
    model: str = JUDGE_MODEL,
):
    prompt = build_single_criterion_prompt(criterion_name, criterion_definition)

    user_input = f"""
Product data:
- product_name: {row['product_name']}
- Product_attribute_list: {row['Product_attribute_list']}
- material: {row['material']}
- warranty: {row['warranty']}

Generated description:
{row['generated_description']}
""".strip()

    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_input},
        ],
        response_format={"type": "json_object"},
    )

    content = response.choices[0].message.content
    return json.loads(content)

In [29]:
criterion_defs = {
    "fluency": "good: The description reads smoothly, flows naturally, and sounds like human-written marketing copy without awkward phrasing or repetition. ok: The description is understandable but contains minor awkward phrasing, slight repetition, or less natural flow. bad: The description is difficult to read, feels robotic, or contains broken phrasing and noticeable repetition.",
    "grammar": "good: No grammatical, spelling, or punctuation errors. Sentences are well-structured and follow standard English conventions. ok: Minor grammatical, spelling, or punctuation errors that do not significantly affect readability or understanding. bad: Frequent or severe grammatical, spelling, or punctuation errors that make the text difficult to read or understand.",
    "tone": "good: Uses a friendly, credible, and persuasive sales voice appropriate for an e-commerce product description. ok: Generally appropriate tone, but somewhat generic, neutral, or less engaging than expected for product marketing. bad: Tone is inappropriate for a product description, such as overly robotic, exaggerated, untrustworthy, or unrelated to a sales context.",
    "length": "good: Between 50 and 90 words (inclusive). ok: Between 40–49 words or 91–110 words. bad: Fewer than 40 words or more than 110 words.",
    "grounding": "good: All information in the description is directly supported by the provided input (name, attributes, material, warranty). No new facts or claims are introduced. ok: Mostly grounded in the provided input, but includes minor unsupported embellishments that do not significantly alter the meaning. bad: Includes unsupported or fabricated information that is not present in the provided input, or contradicts the original data."
}

In [30]:
single_judge_rows = []

for _, row in human_eval_df.iterrows():
    row_result = row.to_dict()

    for criterion, definition in criterion_defs.items():
        try:
            result = run_single_criterion_judge(row, criterion, definition)
            row_result[f"single_{criterion}_verdict"] = result.get("verdict")
            row_result[f"single_{criterion}_explanation"] = result.get("explanation")
        except Exception as e:
            row_result[f"single_{criterion}_verdict"] = None
            row_result[f"single_{criterion}_explanation"] = str(e)

    single_judge_rows.append(row_result)

single_judge_df = pd.DataFrame(single_judge_rows)
single_judge_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,...,single_fluency_verdict,single_fluency_explanation,single_grammar_verdict,single_grammar_explanation,single_tone_verdict,single_tone_explanation,single_length_verdict,single_length_explanation,single_grounding_verdict,single_grounding_explanation
0,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Write a product description based on the follo...,Take your gaming experience to the next level ...,675.51,209,102,82,...,good,"The description reads smoothly and naturally, ...",good,The description is grammatically correct with ...,good,The description uses enthusiastic language lik...,good,"The description is 83 words long, which falls ...",good,The description accurately reflects all the pr...
1,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Write a product description based on the follo...,"""Capture life's moments with the Blink Outdoor...",679.53,207,96,74,...,good,"The description reads smoothly and naturally, ...",good,"The description contains no grammatical, spell...",good,The description uses enthusiastic language lik...,good,"The description is 84 words long, which falls ...",good,The description accurately reflects all the pr...
2,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,Write a product description based on the follo...,Get ready to embark on an epic adventure with ...,780.84,212,111,88,...,good,"The description reads smoothly and naturally, ...",good,"The description contains no grammatical, spell...",good,The description uses enthusiastic language lik...,good,"The description is 88 words long, which falls ...",good,The description accurately reflects all the pr...
3,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,Write a product description based on the follo...,Capture life's most epic moments with the SanD...,676.98,212,93,72,...,good,"The description reads smoothly and naturally, ...",good,"The description contains no grammatical, spell...",good,The description uses enthusiastic and action-o...,good,The description is 80 words long.,good,The description accurately reflects the provid...
4,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,Write a product description based on the follo...,"""Elevate your coffee game with the Keurig K-Su...",681.12,212,108,77,...,good,"The description reads smoothly and naturally, ...",good,The description is grammatically correct with ...,good,The description uses enthusiastic language lik...,good,"The description is 82 words long, which falls ...",good,The description accurately reflects all the pr...


In [31]:
single_agreement = {
    "fluency": (
        single_judge_df["fluency"].str.lower() ==
        single_judge_df["single_fluency_verdict"].str.lower()
    ).mean(),
    "grammar": (
        single_judge_df["grammar"].str.lower() ==
        single_judge_df["single_grammar_verdict"].str.lower()
    ).mean(),
    "tone": (
        single_judge_df["tone"].str.lower() ==
        single_judge_df["single_tone_verdict"].str.lower()
    ).mean(),
    "length": (
        single_judge_df["length"].str.lower() ==
        single_judge_df["single_length_verdict"].str.lower()
    ).mean(),
    "grounding": (
        single_judge_df["grounding"].str.lower() ==
        single_judge_df["single_grounding_verdict"].str.lower()
    ).mean(),
}

single_agreement_df = pd.DataFrame(
    [{"criterion": k, "single_criterion_agreement_rate": v} for k, v in single_agreement.items()]
)

single_agreement_df

,criterion,single_criterion_agreement_rate
0,fluency,1.000000
1,grammar,1.000000
2,tone,0.750000
3,length,0.583333
4,grounding,0.000000


In [32]:
SINGLE_OUTPUT_PATH = BASE_DIR / "outputs" / "assignment_01_single_criterion_judge.xlsx"
single_judge_df.to_excel(SINGLE_OUTPUT_PATH, index=False)

print(f"Saved to: {SINGLE_OUTPUT_PATH}")

Saved to: /Users/mosheoz/Desktop/repos/llm-evaluation-pipeline/outputs/assignment_01_single_criterion_judge.xlsx


## Criterion-by-Criterion Analysis

Evaluating each criterion separately produced mixed results.

- Agreement remained perfect for fluency and grammar, which confirms that the judge is reliable on objective linguistic criteria.
- Tone agreement improved significantly compared to the multi-criterion setting, suggesting that isolating the criterion helped the model focus better on subjective style judgments.
- Length agreement remained moderate, indicating that this criterion is relatively stable and less affected by evaluation context.
- Grounding agreement dropped to 0%, which highlights a fundamental difference between the human evaluator and the judge model.

This result suggests that while isolating criteria reduces interference and improves focus, it does not resolve deeper conceptual differences. In particular, the judge remains much more lenient than the human evaluator when evaluating factual correctness and unsupported claims.

Overall, criterion-by-criterion evaluation improved performance on subjective criteria such as tone, but exposed the judge’s weakness on strict factual evaluation.

---
---

## Final Analysis

### Human Evaluation vs LLM-as-a-Judge

Human evaluation provides more nuanced and context-aware judgments, especially for subtle criteria such as tone and grounding. However, it is slow, expensive, and difficult to scale.

LLM-as-a-judge is fast, scalable, and consistent across large datasets. It applies the rubric in a uniform way, but may be more lenient than a human evaluator, especially for borderline cases.

### Agreement Analysis

The judge achieved very high agreement on objective linguistic criteria such as fluency and grammar, which indicates that LLM-based evaluation is reliable for clear and easy-to-define dimensions.

Agreement was much lower on tone and grounding. This reflects the fact that tone is subjective, and grounding requires strict factual validation. The judge tended to be more lenient than the human evaluator, especially when unsupported claims were minor or presented as marketing language.

### Criterion-by-Criterion Judging

Evaluating one criterion at a time improved agreement on tone, suggesting that isolating criteria can reduce interference and help the judge focus more effectively.

However, grounding agreement remained extremely poor, showing that the problem is not just multi-criterion interference but a deeper limitation in the model’s ability to apply strict factual validation.

### Practical Trade-offs

Human evaluation:
- More accurate and nuanced
- Better at handling subtle edge cases
- Slow, expensive, and hard to scale

LLM-as-a-judge:
- Fast and scalable
- Consistent across large volumes
- Less reliable for strict factual judgments unless carefully tuned

### Recommendation for Production

For a production system that generates thousands of descriptions daily, I would recommend using LLM-as-a-judge as the primary evaluation layer because it is scalable and efficient.

However, I would not rely on it alone for critical criteria such as grounding. A practical production solution should combine:
- LLM-based evaluation for scale
- stricter prompting and rule-based validation for factual constraints
- periodic human review on sampled outputs

This hybrid approach offers a strong balance between scalability, consistency, and reliability.